# 4. Exports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/04_exports.ipynb) [Download the notebook](https://github.com/Khoshkhah/duckOSM/raw/main/docs/notebooks/04_exports.ipynb)

You export the network of Monaco to six formats. Each cell writes the files and does a check on them.

| Format | Function | Check |
|---|---|---|
| SUMO | `to_sumo` | SUMO's `netconvert` reads the plain XML and writes `.net.xml` |
| GMNS | `to_gmns` | `check_gmns`: the rules of the GMNS tables |
| MATSim | `to_matsim` | the XML has one link for each edge |
| OpenDRIVE | `to_opendrive` | the XML has one road for each edge |
| GeoPackage | `to_gis` | DuckDB reads each layer |
| QuackOSM table, PMTiles | `quack`, `duckosm quack --pmtiles` | DuckDB reads the table; the PMTiles header |

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
try:
    import duckosm
except ImportError:
    %pip install -q "duckosm[widget,sumo]" roadstyle
    import duckosm
print("duckosm", duckosm.__version__)

duckosm 0.8.0


In [2]:
area = duckosm.get("Monaco", out="data")      # the OSM file of Monaco and its boundary
db = duckosm.build(pbf=area.pbf, boundary=area.boundary, modes=["driving"],
                   output="data/monaco.duckdb")    # the driving network in one DuckDB file
db

PosixPath('data/monaco.duckdb')

In [3]:
from pathlib import Path


def files(folder):
    """Print the files of a folder and their sizes."""
    for f in sorted(Path(folder).glob("*")):
        print(f"{f.stat().st_size / 1e6:8.2f} MB  {f}")


n_edges = len(duckosm.edges(db))    # the number of driving edges, to compare with each export
n_edges

2840

## SUMO

`to_sumo` writes the plain XML files and runs SUMO's `netconvert` on them. The extra `sumo` installs `netconvert`.

In [4]:
import os, shutil, sys

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]   # netconvert of the extra sumo
sumo = duckosm.to_sumo(db, "out/sumo", run_netconvert=shutil.which("netconvert") is not None)
files("out/sumo")
print(sumo["n_edges"], "edges,", sumo["n_connections"], "connections, netconvert wrote:", sumo.get("net"))

running: netconvert -c out/sumo/network.netccfg


    0.35 MB  out/sumo/network.con.xml
    1.01 MB  out/sumo/network.edg.xml
    5.20 MB  out/sumo/network.net.xml
    0.00 MB  out/sumo/network.netccfg
    0.10 MB  out/sumo/network.nod.xml
2840 edges, 5082 connections, netconvert wrote: out/sumo/network.net.xml


## GMNS

In [5]:
duckosm.to_gmns(db, "out/gmns.duckdb", to_csv="out/gmns")   # a GMNS database and its CSV files
files("out/gmns")
checks = duckosm.check_gmns("out/gmns.duckdb")    # (schema, check, bad rows, rows) for each check
failed = [c for c in checks if c[2] > 0]
print(len(checks), "checks,", len(failed), "with failures")

    0.00 MB  out/gmns/config.csv
    0.00 MB  out/gmns/curb_seg.csv
    0.44 MB  out/gmns/geometry.csv
    0.18 MB  out/gmns/lane.csv
    0.78 MB  out/gmns/link.csv
    0.19 MB  out/gmns/location.csv
    2.01 MB  out/gmns/movement.csv
    0.09 MB  out/gmns/node.csv
    0.00 MB  out/gmns/signal_controller.csv
    0.00 MB  out/gmns/use_definition.csv
    0.00 MB  out/gmns/use_group.csv
    0.01 MB  out/gmns/zone.csv
17 checks, 0 with failures


## MATSim

In [6]:
import gzip
import xml.etree.ElementTree as ET

duckosm.to_matsim(db, "out/matsim_network.xml.gz")
links = ET.parse(gzip.open("out/matsim_network.xml.gz")).findall(".//link")    # the links of the network file
print(f"{Path('out/matsim_network.xml.gz').stat().st_size / 1e6:.2f} MB,", len(links), "links,", n_edges, "edges")

0.10 MB, 2840 links, 2840 edges


## OpenDRIVE (experimental)

In [7]:
duckosm.to_opendrive(db, "out/monaco.xodr")
roads = ET.parse("out/monaco.xodr").findall("road")    # the roads of the OpenDRIVE file
print(f"{Path('out/monaco.xodr').stat().st_size / 1e6:.1f} MB,", len(roads), "roads,", n_edges, "edges")

2.1 MB, 2840 roads, 2840 edges


## GeoPackage

In [8]:
import duckdb

gis = duckosm.to_gis(db, "out/monaco.gpkg", modes=["driving"])
con = duckdb.connect()
con.load_extension("spatial")
# the number of rows of each layer, as DuckDB reads them
{layer: con.execute(f"SELECT count(*) FROM st_read('out/monaco.gpkg', layer='{layer}')").fetchone()[0]
 for layer in gis["layers"]}

{'edges_driving': 2840, 'nodes_driving': 1768, 'boundary': 1}

## QuackOSM table and PMTiles

`quack` reads the OSM file, not the network: it gives all OSM features with their tags, as QuackOSM does. The command `duckosm quack --pmtiles` makes a base map of vector tiles in one file.

In [9]:
quack = duckosm.quack(area.pbf, out="out", name="monaco", formats=["geoparquet"])
# the features of the table, by OSM type
con.sql(f"SELECT split_part(feature_id, '/', 1) AS type, count(*) AS features FROM '{quack[0]}' GROUP BY 1 ORDER BY 1").df()

,type,features
0,node,3996
1,relation,61
2,way,5614


In [10]:
import subprocess

subprocess.run([sys.executable, "-m", "duckosm", "quack", str(area.pbf), "--pmtiles", "-o", "out", "--name", "monaco"],
               check=True)    # the command duckosm quack --pmtiles
header = Path("out/monaco.pmtiles").read_bytes()[:7]    # a PMTiles file starts with "PMTiles"
print(header, f"{Path('out/monaco.pmtiles').stat().st_size / 1e6:.2f} MB")

b'PMTiles' 0.23 MB


     0.0 s  quack --pmtiles: the boundaries
     0.0 s  quack 1: the ways and the relations
     0.1 s  quack 2: the nodes and the points
     0.1 s  quack 3: the ways
     0.1 s  quack 4a: the lines of the relations
     0.1 s  quack 4b: the relations
     0.1 s  quack --pmtiles: the rows of the tiles
     0.1 s  quack --pmtiles: the tiles
quack --pmtiles: 9671 features, 90 tiles, 0.2 MB in 0.10 s
  out/monaco.pmtiles
